# Etapa 2: Análisis Exploratorio de Datos (EDA) y Diagnóstico de Distribuciones

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 2: Data Understanding / EDA) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, DAMA-BOK, PEP 8, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Medidas de Tendencia Central y Dispersión**: Media, mediana, moda, IQR, MAD y desviación estándar.
2. **Medidas de Forma**: Coeficiente de asimetría (*skewness*, $S$) y curtosis de Fisher ($K$).
3. **Diagnóstico Cullen & Frey**: Posicionamiento empírico de las coordenadas ($S^2$ vs $K$) frente a distribuciones teóricas (Normal, Lognormal, Gamma, Weibull).
4. **Diagramas de Caja y Bigotes (*Boxplots*)**: Detección visual de variabilidad y rangos intercuartílicos por año y variedad.
5. **Matrices de Correlación**: Evaluación lineal (Pearson) y monótona (Spearman) entre volúmenes, precios y factores temporales.

In [ ]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    statsmodels>=0.14.0 \
    scikit-learn>=1.3.0 \
    matplotlib>=3.7.0 \
    seaborn>=0.12.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.infrastructure.config import CLEANED_DIR, CURATED_DIR
from src.application.eda_service import EdaService
from src.presentation.cullen_frey_plotter import CullenFreyPlotter
from src.presentation.boxplot_plotter import BoxplotPlotter
from src.presentation.correlation_plotter import CorrelationPlotter

print("Módulos EDA y Renderizadores cargados exitosamente.")

In [ ]:
# ==============================================================================
# CARGA DEL DATASET ÚNICO SIPSA MENSUAL
# ==============================================================================
parquet_path = CLEANED_DIR / "dataset_sipsa_mensual_nacional.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"Dataset cargado exitosamente: {len(df):,} registros mensuales.")
else:
    print(f"AVISO: Archivo {parquet_path} aún no compilado. Generando datos de prueba controlados...")
    np.random.seed(42)
    fechas = pd.date_range("2019-01-01", "2025-12-01", freq="MS")
    variedades = ["PAPA PASTUSA", "PAPA CAPIRO", "PAPA CRIOLLA COLOMBIA"]
    rows = []
    for f in fechas:
        for v in variedades:
            p_base = 2200 if "PASTUSA" in v else (2500 if "CAPIRO" in v else 3200)
            p = float(np.random.gamma(shape=9.0, scale=p_base / 9.0))
            vol = float(np.random.uniform(50, 450))
            rows.append({
                "fecha_mes": f,
                "año": f.year,
                "mes": f.month,
                "divipola_depto": "25",
                "nombre_depto": "CUNDINAMARCA",
                "divipola_mpio": "25875",
                "nombre_mpio": "VILLAPINZON",
                "mercado_mayorista": "BOGOTA, D.C., CORABASTOS",
                "alimento": "PAPA",
                "variedad_papa": v,
                "volumen_ingreso_ton": vol,
                "precio_prom_kg": p,
                "precio_min_kg": p * 0.9,
                "precio_max_kg": p * 1.15,
                "num_transacciones": 30
            })
    df = pd.DataFrame(rows)

eda = EdaService(df)
print("Vista previa de datos listos para EDA:")
display(df.head(5))

In [ ]:
# ==============================================================================
# 1. MEDIDAS DESCRIPTIVAS GLOBALES (TENDENCIA CENTRAL, DISPERSIÓN Y FORMA)
# ==============================================================================
resumen_precios = eda.calcular_medidas_resumen(columna="precio_prom_kg")
resumen_volumen = eda.calcular_medidas_resumen(columna="volumen_ingreso_ton")

df_tabla = pd.DataFrame({
    "Métrica Estadística": list(resumen_precios.keys()),
    "Precio Promedio ($/kg)": [f"{v:,.2f}" if isinstance(v, (int, float)) else str(v) for v in resumen_precios.values()],
    "Volumen Ingresado (Ton)": [f"{v:,.2f}" if isinstance(v, (int, float)) else str(v) for v in resumen_volumen.values()]
})

print("=== RESUMEN GLOBAL DE DISTRIBUCIONES ===")
display(df_tabla)

In [ ]:
# ==============================================================================
# 2. RESUMEN ESTRATIFICADO POR AÑO Y VARIEDAD COMERCIAL
# ==============================================================================
resumen_anual = eda.calcular_resumen_estratificado(col_agrupacion="año", col_variable="precio_prom_kg")
print("=== MÉTRICAS DE PRECIOS POR AÑO ===")
display(resumen_anual[["año", "n_observaciones", "media", "mediana", "desviacion_estandar", "iqr", "asimetria_skewness", "curtosis_fisher_exceso"]])

resumen_variedad = eda.calcular_resumen_estratificado(col_agrupacion="variedad_papa", col_variable="precio_prom_kg")
print("\n=== MÉTRICAS DE PRECIOS POR VARIEDAD ===")
display(resumen_variedad[["variedad_papa", "media", "mediana", "desviacion_estandar", "asimetria_skewness"]])

In [ ]:
# ==============================================================================
# 3. DIAGNÓSTICO DE CULLEN Y FREY (SKEWNESS^2 VS KURTOSIS)
# ==============================================================================
coords_cf = eda.calcular_coordenadas_cullen_frey(columna="precio_prom_kg", n_bootstraps=300)

print(f"Asimetría observada: {coords_cf['skewness_observado']:.3f}")
print(f"Asimetría al cuadrado (S^2): {coords_cf['skewness_sq_observado']:.3f}")
print(f"Curtosis de Pearson (K): {coords_cf['kurtosis_observado']:.3f}")

out_cf = CURATED_DIR / "cullen_frey_precios_sipsa.png"
fig_cf = CullenFreyPlotter.plot(
    cf_coords=coords_cf,
    titulo="Diagnóstico Cullen & Frey — Precios Mayoristas de la Papa (2019-2025)",
    output_path=out_cf
)
plt.show()

In [ ]:
# ==============================================================================
# 4. DIAGRAMAS DE CAJA Y BIGOTES (BOXPLOTS)
# ==============================================================================
out_box = CURATED_DIR / "boxplot_precios_anuales.png"
fig_box = BoxplotPlotter.plot_comparativo(
    df=df,
    x_col="año",
    y_col="precio_prom_kg",
    hue_col="variedad_papa",
    titulo="Dispersión Anual de Precios por Variedad de Papa",
    xlabel="Año Calendario",
    ylabel="Precio Promedio ($/kg)",
    output_path=out_box
)
plt.show()

In [ ]:
# ==============================================================================
# 5. MATRICES DE CORRELACIÓN BIVARIADA: PEARSON VS SPEARMAN
# ==============================================================================
corr_p, corr_s = eda.calcular_matrices_correlacion()

out_corr = CURATED_DIR / "matrices_correlacion_pearson_spearman.png"
fig_corr = CorrelationPlotter.plot_dual_heatmaps(
    corr_pearson=corr_p,
    corr_spearman=corr_s,
    titulo="Correlación Bivariada entre Volúmenes, Precios y Tiempo",
    output_path=out_corr
)
plt.show()

### Hallazgos Principales de la Etapa 2 (EDA):
1. **Asimetría Positiva y Curtosis Elevada**: Las series de precios presentan un sesgo hacia la derecha ($S > 0$) y leptocurtosis ($K > 3$), lo que sitúa la masa empírica en el área Gamma / Lognormal del gráfico de Cullen & Frey.
2. **Justificación del Enfoque No Paramétrico**: La no normalidad comprobada empíricamente justifica formalmente la aplicación del canal estadístico no paramétrico (Kruskal-Wallis, Fligner-Killeen y Bootstrap BCa).
3. **Transición a la Etapa 3 (Validation)**: Los datos pasan a verificación estricta de reglas de negocio y consistencia DIVIPOLA.